# Exploratory Data Analysis: MIMIC-IV Readmission Cohort

Two jobs here. Extract the study cohort from MIMIC-IV, then explore it before any
modelling. The outcome of interest throughout is whether a patient is readmitted within
30 days of discharge.

Nothing in this notebook identifies an individual. Every number, table and chart
describes the group as a whole, for example that 20.6% of admissions were followed by a
readmission.

MIMIC-IV is credentialed data under a PhysioNet data use agreement. Keep the extracted
files private and delete them once the work is finished.

## Part 1: Extracting the cohort

### Libraries

Packages for querying the database and for plotting. Runs once per session.

In [73]:
%pip install -q google-cloud-bigquery db-dtypes pyarrow matplotlib seaborn

### Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and
delete the data once the work is finished.

In [ ]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

### Connecting to the database

MIMIC-IV is hosted on BigQuery, and access requires an account approved for this
specific dataset. Queries are billed to a separate cloud project rather than to the one
that hosts the data.

In [74]:
# The Google Cloud project that BigQuery bills each query to.
# This is NOT the same as physionet-data (below), which is just where the data lives.
GCP_PROJECT = "predictive-keep-340709"  # replace with your own project ID if different

# Hosted and local environments sign in to Google in different ways.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Interactive sign-in when running in a hosted environment.
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated.")
else:
    # Locally, sign-in happens once in advance via
    # `gcloud auth application-default login`, so nothing is needed here.
    print("Assuming `gcloud auth application-default login` has already been run.")

### The extraction query

The cohort is the specific set of hospital admissions this study uses. Not every
admission in MIMIC-IV is included.

The query returns one row per admission, so a patient admitted several times appears
several times, and works out for each admission whether that patient returned within 30
days of discharge.

Three exclusions are applied, each explained in the comments in the query:

- Admissions ending in death are removed, since a patient who died cannot be readmitted.
- 45 admissions have a negative length of stay, where discharge is recorded before
  admission. These are data entry errors.
- Age is capped at 91. MIMIC-IV already masks the age of anyone over 89, and the age
  calculation used here can push that masked value higher, so the cap keeps every age
  inside the already anonymised range.

In [75]:
from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project=GCP_PROJECT)

HOSP = "physionet-data.mimiciv_3_1_hosp"  # hospital-wide admissions/patient tables
ICU = "physionet-data.mimiciv_3_1_icu"    # intensive care unit stay tables

QUERY = f"""
-- Step A: for every admission, work out the date of that same patient's NEXT
-- admission (if any), so we can later check whether it happened within 30 days.
WITH ordered AS (
  SELECT
    a.subject_id,
    a.hadm_id,
    a.admittime,
    a.dischtime,
    a.admission_type,
    a.admission_location,
    a.insurance,
    a.marital_status,
    a.race,
    a.language,
    a.hospital_expire_flag,   -- 1 if the patient died during this admission
    p.gender,
    p.anchor_age,             -- the patient's age in a fixed reference year (see below)
    p.anchor_year,            -- that reference year
    LEAD(a.admittime) OVER (PARTITION BY a.subject_id ORDER BY a.admittime) AS next_admittime
  FROM `{HOSP}.admissions` a
  JOIN `{HOSP}.patients` p USING (subject_id)
),
-- Step B: flag which admissions included a stay in the Intensive Care Unit (ICU).
icu_flag AS (
  SELECT DISTINCT hadm_id, TRUE AS had_icu_stay
  FROM `{ICU}.icustays`
)
-- Step C: assemble the final table - one row per admission, with the target
-- outcome and every feature used later in the dissertation.
SELECT
  o.subject_id,
  o.hadm_id,
  -- Age at the time of THIS admission, derived from the fixed reference age/year
  -- above, capped at 91 for the de-identification reason explained above.
  LEAST(o.anchor_age + (EXTRACT(YEAR FROM o.admittime) - o.anchor_year), 91) AS age_at_admission,
  o.gender,
  o.admission_type,
  o.admission_location,
  o.insurance,
  o.marital_status,
  o.race,
  o.language,
  -- How long the admission lasted, in days.
  DATETIME_DIFF(o.dischtime, o.admittime, HOUR) / 24.0 AS length_of_stay_days,
  COALESCE(icu.had_icu_stay, FALSE) AS had_icu_stay,
  -- The target outcome: 1 if the next admission (if any) started within 30 days
  -- of this admission's discharge date, otherwise 0.
  CASE WHEN DATETIME_DIFF(o.next_admittime, o.dischtime, DAY) <= 30 THEN 1 ELSE 0 END AS readmitted_30d
FROM ordered o
LEFT JOIN icu_flag icu USING (hadm_id)
WHERE o.hospital_expire_flag = 0                                    -- exclude deaths
  AND DATETIME_DIFF(o.dischtime, o.admittime, HOUR) >= 0             -- exclude negative length of stay
"""

print("Running extraction query against MIMIC-IV on BigQuery...")
df = client.query(QUERY).to_dataframe()
print(f"Extracted {len(df)} admissions, covering {df['subject_id'].nunique()} unique patients.")
print(f"Share of admissions followed by a readmission within 30 days: {df['readmitted_30d'].mean():.2%}")

### Saving the cohort

Written to a parquet file so the rest of this notebook and the later steps can reuse it
without querying the database again.

In [76]:
out_dir = Path("data")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "cohort.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved to {out_path.resolve()}")

## Part 2: Exploring the cohort

Before modelling anything it is worth knowing what the real data looks like: whether it
is clean, who is in it, and how each characteristic relates to readmission.

Everything below is a summary describing the whole set of admissions, so it is safe to
reuse in the write-up. Each chart is saved individually to figures/ with readable titles
and category names rather than raw column codes, and is followed by a short note on how
to read it.

### Charting helpers

Shared plotting code, so every chart uses the same style and the same readable labels.
COLUMN_LABELS turns a column name into a chart title and VALUE_LABELS does the same for
coded values such as F and M. The two plotting functions cover the two chart types used
repeatedly below: how common each category is, and the readmission rate within each
category.

In [77]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.15)  # a clean, readable default chart style
fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

TARGET = "readmitted_30d"
NUMERIC_COLS = ["age_at_admission", "length_of_stay_days"]
CATEGORICAL_COLS = [
    "gender",
    "admission_type",
    "admission_location",
    "insurance",
    "marital_status",
    "race",
    "language",
    "had_icu_stay",
]

# Plain-English chart titles/axis labels for each column.
COLUMN_LABELS = {
    "gender": "Gender",
    "admission_type": "Admission Type",
    "admission_location": "Admission Location",
    "insurance": "Insurance",
    "marital_status": "Marital Status",
    "race": "Race",
    "language": "Language",
    "had_icu_stay": "ICU Stay",
    "age_at_admission": "Age at Admission",
    "length_of_stay_days": "Length of Stay (days)",
}

# Plain-English labels for coded VALUES within a column - only needed for columns
# whose raw values are abbreviations or True/False rather than readable text
# (columns like race or admission_type already store full-text category names).
VALUE_LABELS = {
    "gender": {"F": "Female", "M": "Male"},
    "had_icu_stay": {True: "ICU Stay", False: "No ICU Stay"},
}


def relabel_values(series: pd.Series, col: str) -> pd.Series:
    """Swap coded values (e.g. 'F') for their plain-English label (e.g. 'Female'),
    for any column that needs it. Columns without a mapping are returned unchanged."""
    mapping = VALUE_LABELS.get(col)
    if mapping is None:
        return series
    return series.map(mapping).fillna(series.astype(str))


def plot_top_categories(series: pd.Series, col: str, ax, top_n: int = 10, title: str = "") -> None:
    """Bar chart: what share of admissions falls into each category of this column?
    Only the top_n most common categories are shown individually - anything smaller
    is grouped into a single 'Other' bar, so the chart stays readable even for
    columns with many rare categories (e.g. race, language)."""
    vc = relabel_values(series, col).value_counts(normalize=True, dropna=False)
    if len(vc) > top_n:
        top = vc.iloc[:top_n]
        other_share = vc.iloc[top_n:].sum()
        vc = pd.concat([top, pd.Series({f"Other ({len(vc) - top_n} categories)": other_share})])
    vc.plot(kind="bar", ax=ax, color="#4C72B0")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Proportion of Admissions")
    ax.tick_params(axis="x", rotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")


def plot_target_rate_by_category(frame: pd.DataFrame, col: str, ax, top_n: int = 10, title: str = "") -> None:
    """Bar chart: within each category of this column, what proportion of admissions
    were followed by a readmission within 30 days? Restricted to the top_n most
    frequent categories, because a rate calculated from only a handful of admissions
    is unreliable and would clutter the chart."""
    working = frame[[col, TARGET]].copy()
    working[col] = relabel_values(working[col], col)
    top_categories = working[col].value_counts().iloc[:top_n].index
    subset = working[working[col].isin(top_categories)]
    rates = subset.groupby(col)[TARGET].mean().sort_values(ascending=False)
    rates.plot(kind="bar", ax=ax, color="#C44E52")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("30-Day Readmission Rate")
    ax.tick_params(axis="x", rotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")


def save_and_show(fig, name: str) -> None:
    """Save a chart as a PNG file in figures/ and display it in the notebook."""
    fig.tight_layout()
    fig.savefig(fig_dir / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

### Dataset summary

Size of the cohort, and how common readmission actually is.

In [78]:
summary = pd.DataFrame(
    {
        "Metric": ["Admissions", "Unique patients", "30-day readmission rate"],
        "Value": [
            f"{len(df):,}",
            f"{df['subject_id'].nunique():,}",
            f"{df['readmitted_30d'].mean():.1%}",
        ],
    }
)
summary

### Columns, types and missingness

Every column extracted, its type, how much of it is missing, and how many distinct
values it takes. Missing data and unexpected variety in a column both cause problems
later on.

In [79]:
feature_summary = pd.DataFrame(
    {
        "Column": df.columns,
        "Type": df.dtypes.astype(str).values,
        "% missing": (df.isna().mean() * 100).round(2).values,
        "n_unique": [df[c].nunique(dropna=False) for c in df.columns],
    }
)
feature_summary

### Data quality checks

Two checks following from the exclusions applied during extraction. Age should never
exceed 91, and length of stay should never be negative. Both are asserted rather than
just printed, so a broken assumption stops the notebook instead of quietly producing
wrong results further down.

In [80]:
print("Age at admission:")
print(df["age_at_admission"].describe())
assert df["age_at_admission"].max() <= 91, "age_at_admission exceeds the 91 cap - check the query"
n_at_ceiling = (df["age_at_admission"] >= 91).sum()
print(f"\nAdmissions at the age-91 de-identification ceiling: {n_at_ceiling} ({n_at_ceiling / len(df):.2%})")

print("\nLength of stay (days):")
print(df["length_of_stay_days"].describe())
assert (df["length_of_stay_days"] >= 0).all(), "negative LOS present - check the query's WHERE clause"
n_zero = (df["length_of_stay_days"] == 0).sum()
print(f"\nZero-day stays (same-day discharge, clinically plausible - kept): {n_zero}")

### Category counts

Some columns take two values, others take dozens. This affects how the columns are
encoded for modelling, and columns with many rare categories are harder for a generator
to reproduce accurately.

In [81]:
cardinality = pd.DataFrame(
    {"column": CATEGORICAL_COLS, "n_categories": [df[c].nunique(dropna=False) for c in CATEGORICAL_COLS]}
).sort_values("n_categories", ascending=False)
cardinality

In [82]:
# List, for each column, any category that makes up less than 1% of admissions.
# These "rare categories" are candidates for grouping into a single "Other" bucket
# later on, since a handful of examples isn't enough to learn a reliable pattern from.
print("Categories under 1% share (candidates for grouping into 'Other' at preprocessing time):\n")
for col in CATEGORICAL_COLS:
    vc = df[col].value_counts(normalize=True, dropna=False)
    rare = vc[vc < 0.01]
    if len(rare):
        print(f"{col} ({len(rare)} rare of {len(vc)} total): {', '.join(str(x) for x in rare.index)}\n")

### Choosing the prediction task

Two outcomes were considered: in-hospital mortality, and readmission within 30 days. The
chart compares how common each one is.

In [83]:
# Mortality rate has to be measured on the FULL, unfiltered admissions table -
# our cohort (df) has already excluded deaths, so it can't be measured from df.
mortality_query = f"SELECT ROUND(AVG(hospital_expire_flag), 4) AS mortality_rate FROM `{HOSP}.admissions`"
mortality_rate = client.query(mortality_query).to_dataframe()["mortality_rate"].iloc[0]
readmission_rate = df["readmitted_30d"].mean()

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    ["In-Hospital Mortality", "30-Day Readmission"],
    [mortality_rate, readmission_rate],
    color=["#8C8C8C", "#4C72B0"],
)
ax.set_ylabel("Rate")
ax.set_title("Candidate Prediction Tasks: Class Balance Comparison")
for bar, rate in zip(bars, [mortality_rate, readmission_rate]):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.01, f"{rate:.1%}", ha="center")
save_and_show(fig, "task_selection_mortality_vs_readmission")

Mortality is far rarer than readmission. Both a classifier and a generator struggle
more with an outcome occurring in about 2% of cases than one occurring in about 20%,
simply because there are fewer examples to learn from. That is why readmission is the
primary task.

## Univariate distributions

One characteristic at a time, without yet relating it to the outcome. This builds a
picture of who is in the cohort before the next section looks at relationships.

In [84]:
rates = df["readmitted_30d"].value_counts(normalize=True).sort_index()

fig, ax = plt.subplots(figsize=(5, 5))
ax.bar(["Not Readmitted", "Readmitted \u2264 30 Days"], rates.values, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("Proportion of Admissions")
ax.set_title("Class Balance: 30-Day Readmission")
for i, v in enumerate(rates.values):
    ax.text(i, v + 0.01, f"{v:.1%}", ha="center")
save_and_show(fig, "class_balance")

The share of admissions followed by a readmission, against those that were not. An
imbalanced outcome is harder to model and has to be evaluated with that in mind, which
the methodology chapter covers.

In [85]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(df["age_at_admission"], bins=30, ax=ax, color="#4C72B0")
ax.set_xlabel("Age at Admission (years, capped at 91)")
ax.set_ylabel("Number of Admissions")
ax.set_title("Age Distribution")
save_and_show(fig, "age_distribution")

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="readmitted_30d", y="age_at_admission", ax=ax, palette=["#4C72B0", "#DD8452"])
ax.set_xticklabels(["Not Readmitted", "Readmitted ≤30d"])
ax.set_xlabel("")
ax.set_ylabel("Age at Admission (years)")
ax.set_title("Age by Readmission Outcome")
save_and_show(fig, "age_by_readmission_outcome")

The first chart is the overall spread of ages. The second splits age by outcome, giving
an early sense of whether age looks related to readmission. The correlation matrix later
checks this more formally.

In [86]:
# Length of stay is heavily right-skewed (a small number of very long admissions) -
# clipped at 30 days here purely for readability of the plot, not for modeling.
los_clipped = df["length_of_stay_days"].clip(upper=30)

fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(los_clipped, bins=30, ax=ax, color="#55A868")
ax.set_xlabel("Length of Stay (days, clipped at 30)")
ax.set_ylabel("Number of Admissions")
ax.set_title("Length of Stay Distribution")
save_and_show(fig, "length_of_stay_distribution")

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(
    data=df.assign(los_clipped=los_clipped),
    x="readmitted_30d",
    y="los_clipped",
    ax=ax,
    palette=["#4C72B0", "#DD8452"],
)
ax.set_xticklabels(["Not Readmitted", "Readmitted ≤30d"])
ax.set_xlabel("")
ax.set_ylabel("Length of Stay (days, clipped at 30)")
ax.set_title("Length of Stay by Readmission Outcome")
save_and_show(fig, "length_of_stay_by_readmission_outcome")

The same pair of charts for length of stay. It is heavily skewed, since most admissions
are short and a few last a very long time, so the axis is cut at 30 days for
readability. The cut applies only to the chart, never to a calculation.

In [87]:
# One clean, individually-saved figure per categorical feature - not a packed grid -
# so each one is legible on its own when placed in the dissertation.
for col in CATEGORICAL_COLS:
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_top_categories(df[col], col, ax, title=f"Distribution: {COLUMN_LABELS[col]}")
    save_and_show(fig, f"distribution_{col}")

One bar chart per remaining characteristic, showing what share of admissions falls into
each category. The same chart repeated eight times rather than eight different chart
types.

## Relationships with the outcome

Each characteristic alongside readmission. More informative than the univariate charts,
because it shows directly whether the readmission rate changes with, say, admission type
or insurance.

In [88]:
for col in CATEGORICAL_COLS:
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_target_rate_by_category(df, col, ax, title=f"30-Day Readmission Rate by {COLUMN_LABELS[col]}")
    save_and_show(fig, f"readmission_rate_by_{col}")

Each bar is the readmission rate within one category. The bar for Emergency shows the
proportion of emergency admissions followed by a readmission, not the proportion of all
admissions that were emergencies. This is also what justifies including a characteristic
as a model input: if the rate barely moves across its categories, it carries little
information.

## Repeat admissions

Many patients appear more than once. This matters for how the data is later split. If
one patient's admissions appeared in both the training and test sets, a model would be
tested on someone it had already seen, which inflates apparent performance. That is data
leakage.

In [89]:
adm_counts = df["subject_id"].value_counts()
print(f"Mean admissions per patient: {adm_counts.mean():.2f}")
print(f"Median: {adm_counts.median():.0f} | Max: {adm_counts.max()}")
print(f"Patients with more than one admission: {(adm_counts > 1).sum():,} ({(adm_counts > 1).mean():.1%})")

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.histplot(adm_counts.clip(upper=10), bins=10, ax=ax, color="#55A868")
ax.set_xlabel("Admissions per Patient (clipped at 10 for readability)")
ax.set_ylabel("Number of Patients")
ax.set_title("Admissions per Patient")
save_and_show(fig, "admissions_per_patient")

How many patients had one admission, two, three and so on, with anything above ten
grouped into the last bar. Patients average 2.45 admissions, pulled up by a minority with
many, and 45.1% have more than one. This is the evidence for splitting by patient rather
than by admission.

## Correlations between numeric features

A correlation summarises how strongly two numeric measurements move together, on a scale
from -1 to +1. Near 0 means no straight-line relationship. It only applies to numeric
columns, which is why the categorical ones were examined separately above.

In [90]:
corr_df = df[["age_at_admission", "length_of_stay_days", "readmitted_30d"]].copy()
corr_df["had_icu_stay"] = df["had_icu_stay"].astype(int)
corr_df.columns = ["Age at Admission", "Length of Stay", "30-Day Readmission", "ICU Stay"]
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation Matrix: Numeric Features and Target")
save_and_show(fig, "correlation_heatmap")

Each cell is the correlation between a row and a column, so the diagonal is always 1.
These numeric correlations are weak on their own. The readmission-rate charts above show
where the categorical signal actually sits, which a correlation between numeric columns
cannot capture.

## Downloading the figures

In [95]:
if IN_COLAB:
    import base64
    import shutil
    from IPython.display import HTML, display

    zip_path = shutil.make_archive("figures", "zip", fig_dir)
    with open(zip_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode()

    n_charts = len(list(fig_dir.glob("*.png")))
    display(HTML(
        f'<a download="figures.zip" href="data:application/zip;base64,{encoded}" '
        f'style="font-size:16px;">Click here to download figures.zip ({n_charts} charts)</a>'
    ))